# Graduate Admission Prediction — Ensemble Classification

Academic practical using classification models to predict the binary `Chance of Admit` target from applicant profile and academic features.

In [ ]:
import pandas as pd 
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
import seaborn as sns
from warnings import filterwarnings
filterwarnings('ignore')
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler 
from sklearn.utils import resample
from sklearn.utils import shuffle
from sklearn import metrics
from sklearn.metrics import classification_report
from sklearn.ensemble import AdaBoostClassifier
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score
from sklearn.metrics import roc_curve
from sklearn.metrics import roc_auc_score
from sklearn.metrics import confusion_matrix
from sklearn.model_selection import cross_val_score
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
import random

In [ ]:
# load the csv file
df_heart = pd.read_csv("Uad.xls")

# display the first two observations
df_heart.head()

In [ ]:
# check the size of data
df_heart.shape

In [ ]:
# check data types of variables
df_heart.dtypes

In [ ]:
df_heart.isnull().sum()

In [ ]:
# Drops columns from index 0 up to (but not including) index 3
df_heart = df_heart.drop(df_heart.columns[0], axis=1)



In [ ]:
df_heart['Research'] = df_heart['Research'].astype(object)


In [ ]:
df_heart['Research'] = pd.get_dummies(data = df_heart['Research'], drop_first=True)

In [ ]:
df_heart.T

In [ ]:
# set the plot size using 'rcParams'
# once the plot size is set using 'rcParams', it sets the size of all the forthcoming plots in the file
# pass width and height in inches to 'figure.figsize'
plt.rcParams['figure.figsize'] = [10,5]

In [ ]:
# consider only the target variable
df_target = df_heart['Chance of Admit']

# get counts of 0's and 1's in the 'Loan_Status' variable
df_target.value_counts()

# plot the countplot of the variable 'Loan_Status'
sns.countplot(x = df_target)

# use below code to print the percentage values in the graph
# 'x' and 'y' gives position of the text
# 's' is the text to be printed
plt.text(x = -0.09, y = df_target.value_counts()[0] - 100, s = str(round((df_target.value_counts()[0])*100/len(df_target),2)) + '%')
plt.text(x = 0.90, y = df_target.value_counts()[1] - 100, s = str(round((df_target.value_counts()[1])*100/len(df_target),2)) + '%')

# add plot and axes labels
# set text size using 'fontsize'
plt.title('Count Plot for Target Variable (Chance of Admit)', fontsize = 15)
plt.xlabel('Target Variable', fontsize = 15)
plt.ylabel('Count', fontsize = 15)

# to show the plot
plt.show()

## Random Forest

In [ ]:
df_heart

In [ ]:
y = df_heart['Chance of Admit']
X = df_heart.drop('Chance of Admit',axis=1)
X.info()

In [ ]:
X_train,X_test,y_train,y_test=train_test_split(X,y,random_state=50,test_size=0.2)

## Hyperparameter tuning

In [ ]:
tuned_params=[{'criterion':['entropy', 'gini'],'n_estimators':[90, 100, 150, 200],'max_depth':[10, 15, 20],'min_samples_split':[2, 5, 8]}]
rf_classify=RandomForestClassifier(random_state=100,oob_score=True)
rf_grid=GridSearchCV(estimator=rf_classify,param_grid=tuned_params,cv=5).fit(X_train,y_train)
print('Best Parameters:',rf_grid.best_params_,'\n')

In [ ]:
rf_tuned_model=RandomForestClassifier(criterion='entropy',max_depth=10,min_samples_split=8,n_estimators=100,random_state=100)
scores=cross_val_score(estimator=rf_tuned_model,X=X_train,y=y_train,cv=10,scoring='roc_auc')
print('Mean ROC-AUC Score:',scores.mean())

## Interpretation
A tuned Random Forest model was evaluated using 10-fold cross-validation with ROC-AUC as the performance metric. The mean ROC-AUC score obtained was 0.9249, indicating excellent classification performance. This means that the model has approximately a 92.49% probability of correctly ranking a randomly selected admitted student higher than a randomly selected non-admitted student. The high ROC-AUC value suggests that the tuned Random Forest model generalizes well and is suitable for predicting graduate admission outcomes.

In [ ]:
rf_tuned_model=RandomForestClassifier(criterion='entropy',max_depth=10,min_samples_split=8,n_estimators=100,random_state=100).fit(X_train,y_train)
y_pred=rf_tuned_model.predict(X_test)
print(classification_report(y_test,y_pred))

## Interpretation
The Random Forest model achieved an overall accuracy of 90% on the test dataset. For the non-admitted class, the model obtained a precision of 89%, recall of 93%, and F1-score of 91%. For the admitted class, the precision was 91%, recall was 86%, and F1-score was 89%. The macro and weighted averages were both 90%, indicating balanced performance across classes. Therefore, the model is effective in distinguishing between admitted and non-admitted applicants and can be considered a reliable admission prediction model.

In [ ]:
rf_tuned_model=RandomForestClassifier(criterion='gini',max_depth=15,min_samples_split=2,n_estimators=90,random_state=100).fit(X_train,y_train)
important_features=pd.DataFrame({'Features':X_train.columns,'Importance':rf_tuned_model.feature_importances_})
important_features=important_features.sort_values('Importance',ascending=False)
important_features


## Confusion Matrix

In [ ]:
cm=confusion_matrix(y_test,y_pred)
conf_matrix=pd.DataFrame(data=cm,columns=['Predict Chance of Admit:No','Predict Chance of Admit:Yes'],index=['Actual Chance of Admit:No','Actual Chance of Admit:Yes'])
sns.heatmap(conf_matrix,annot=True,fmt='d',cmap=ListedColormap(['lightskyblue']),cbar=False,linewidths=0.2)
plt.show()

In [ ]:
tn=cm[0][0]
tp=cm[1][1]
fp=cm[0][1]
fn=cm[1][0]
total=tn+tp+fp+fn
correct_classify=((tn+tp)/total)*100
print('Correct Classify:',correct_classify)
incorrect_classify=((fp+fn)/total)*100
print('Incorrect Classify:',incorrect_classify)
specificity=tn/(tn+fp)
print('Specificity:',specificity)
sensitivity=tp/(tp+fn)
print('sensitivity:',sensitivity)

## Interpretation
The model achieved an accuracy of 91.25%, indicating that it correctly classified most applicants. The sensitivity of 89.19% shows that the model successfully identified nearly 89% of admitted students, while the specificity of 93.02% indicates that it correctly identified over 93% of non-admitted applicants. The low error rate of 8.75% suggests strong predictive performance. Overall, the model is reliable for admission prediction and performs slightly better in recognizing non-admitted applicants than admitted applicants.

In [ ]:
fpr,tpr,threshold=roc_curve(y_test,y_pred)
plt.plot([0,1],[0,1],'r--')
plt.plot(fpr,tpr)
plt.text(x=0.02,y=0.8,s=('AUC Score:',metrics.roc_auc_score(y_test,y_pred)))
plt.grid(True)
plt.show()

## Interpretation
The Random Forest model achieved an AUC score of 0.9111, indicating excellent classification performance. Since the AUC value is greater than 0.90, the model has a strong ability to distinguish between admitted and non-admitted applicants. This suggests that the model can effectively rank applicants according to their likelihood of admission.

## Adaboost

## Hyperparameter tuning

In [ ]:
tune_params=[{'n_estimators':[100, 120, 150, 200, 250, 300, 350],'learning_rate':[0.1, 0.01, 0.001, 0.15, 0.015]}]
ada_model=AdaBoostClassifier(random_state=50)
ada_grid=GridSearchCV(estimator=ada_model,param_grid=tune_params,cv=5,n_jobs=-1).fit(X_train,y_train)
print('Best parameters for AdaBoost:',ada_grid.best_params_,'\n')

In [ ]:
X_train,X_test,y_train,y_test=train_test_split(X,df_target,random_state=50,test_size=0.2)
ada_model=AdaBoostClassifier(learning_rate = 0.001,n_estimators=100,random_state=50).fit(X_train,y_train)
y_pred_full=ada_model.predict(X_test)
print(classification_report(y_test,y_pred_full))

## Interpretation
The AdaBoost classifier achieved an overall accuracy of 85%. The model obtained precision values of 85% for both classes. The recall values were 89% for non-admitted applicants and 81% for admitted applicants. The corresponding F1-scores were 87% and 83%, respectively. These results indicate that the AdaBoost model provides good classification performance; however, its predictive accuracy is lower than that of the Random Forest model. Therefore, Random Forest remains the preferred model for graduate admission prediction.

In [ ]:
fpr,tpr,thresholds=roc_curve(y_test,y_pred_full)
plt.plot([0,1],[0,1],'r--')
plt.plot(fpr,tpr)
plt.text(x=0.02,y=0.8,s=('AUC:',metrics.roc_auc_score(y_test,y_pred_full)))
plt.grid(True)

In [ ]:
ada_model=AdaBoostClassifier(learning_rate = 0.001,n_estimators=100,random_state=50)
scores=cross_val_score(estimator=ada_model,X=X_train,y=y_train,cv=10,scoring='roc_auc')
print('Mean ROC-AUC Score:',scores.mean())

## Interpretation
The AdaBoost model achieved a mean ROC-AUC score of 0.8797 using 10-fold cross-validation. This indicates that the model has good discriminative ability and can effectively distinguish between admitted and non-admitted applicants. The score suggests that the model generalizes well to unseen data and provides reliable classification performance.

In [ ]:
cm=confusion_matrix(y_test,y_pred_full)
conf_matrix=pd.DataFrame(data=cm,columns=['Pred Chance of Admit:No','Pred Chance of Admit:Yes'],index=['Act Chance of Admit:No','Act Chance of Admit:Yes'])
sns.heatmap(conf_matrix,annot=True,fmt='d',cmap=ListedColormap(['lightskyblue']),cbar=False,linewidth=0.2)
plt.show()

In [ ]:
tn=cm[0][0]
tp=cm[1][1]
fp=cm[0][1]
fn=cm[1][0]
total=tn+tp+fp+fn
correct_classify=((tn+tp)/total)*100
print('Correct Classify:',correct_classify)
incorrect_classify=((fp+fn)/total)*100
print('Incorrect Classify:',incorrect_classify)
specificity=tn/(tn+fp)
print('Specificity:',specificity)
sensitivity=tp/(tp+fn)
print('sensitivity:',sensitivity)

## Gradient Boosting

# Hyperparameter Tuning

In [ ]:
tune_params=[{'n_estimators':[100, 120, 150, 200, 250, 300, 350],'learning_rate':[0.1, 0.01, 0.001, 0.15, 0.015],'max_depth':[2,3,4,5,6]}]
gb_model=GradientBoostingClassifier(random_state=10)
gb_grid=GridSearchCV(estimator=gb_model,param_grid=tune_params,cv=5,n_jobs=-1).fit(X_train,y_train)
print('Best parameters for GBM:',gb_grid.best_params_,'\n')

In [ ]:
gb_model=GradientBoostingClassifier(n_estimators=250,learning_rate=0.1,max_depth=2,random_state=10).fit(X_train,y_train)
y_pred_gb=gb_model.predict(X_test)
print('Classification Report GB: ',classification_report(y_test,y_pred_gb))

In [ ]:
fpr,tpr,thresholds=roc_curve(y_test,y_pred_gb)
plt.plot(fpr,tpr)
plt.text(x=0.02,y=0.8,s=('AUC:',metrics.roc_auc_score(y_test,y_pred_gb)))
plt.grid(True)

In [ ]:
gb_model=GradientBoostingClassifier(n_estimators=250,learning_rate=0.1,max_depth=2,random_state=10)
scores=cross_val_score(estimator=ada_model,X=X_train,y=y_train,cv=10,scoring='roc_auc')
print('Mean ROC-AUC Score:',scores.mean())

In [ ]:
cm=confusion_matrix(y_test,y_pred_gb)
conf_matrix=pd.DataFrame(data=cm,columns=['Pred Chance of Admit:No','Pred Chance of Admit:Yes'],index=['Act Chance of Admit:No','Act Chance of Admit:Yes'])
sns.heatmap(conf_matrix,annot=True,fmt='d',cmap=ListedColormap(['lightskyblue']),cbar=False,linewidth=0.2)
plt.show()

In [ ]:
tn=cm[0][0]
tp=cm[1][1]
fp=cm[0][1]
fn=cm[1][0]
total=tn+tp+fp+fn
correct_classify=((tn+tp)/total)*100
print('Correct Classify:',correct_classify)
incorrect_classify=((fp+fn)/total)*100
print('Incorrect Classify:',incorrect_classify)
specificity=tn/(tn+fp)
print('Specificity:',specificity)
sensitivity=tp/(tp+fn)
print('sensitivity:',sensitivity)

## XG Boost

## Hyperparameter Tuning

In [ ]:
tune_params=[{'n_estimators':[100,120,150],'learning_rate':[0.1, 0.01, 0.001, 0.15, 0.015],'gamma':[2,3,4,5,6],'max_depth':[2,3,4,5,6]}]
xgb_model=XGBClassifier(random_state=10,n_jobs=-1)
xgb_grid=GridSearchCV(estimator=xgb_model,param_grid=tune_params,cv=5,n_jobs=-1).fit(X_train,y_train)
print('Best parameters for XGB Model:',xgb_grid.best_params_,'\n')


In [ ]:
xgb_model=XGBClassifier(learning_rate=0.01,gamma=2).fit(X_train,y_train)
y_pred_xgb=xgb_model.predict(X_test)
print('Classification Report for XGB: ',classification_report(y_test,y_pred_xgb))

In [ ]:
fpr,tpr,thresholds=roc_curve(y_test,y_pred_xgb)
plt.plot([0,1],[0,1],'r--')
plt.plot(fpr,tpr)
plt.text(x=0.02,y=0.8,s=('AUC:',metrics.roc_auc_score(y_test,y_pred_xgb)))
plt.grid(True)

In [ ]:
xgb_model=XGBClassifier(n_estimators=100,learning_rate=0.15,gamma=2,max_depth=6,random_state=10).fit(X_train,y_train)
y_pred_xgb=xgb_model.predict(X_test)
cm=confusion_matrix(y_test,y_pred_xgb)
conf_matrix=pd.DataFrame(data=cm,columns=['Pred Chance of Admit:No','Pred Chance of Admit:Yes'],index=['Act Chance of Admit:No','Act Chance of Admit:Yes'])
sns.heatmap(conf_matrix,annot=True,fmt='d',cmap=ListedColormap(['lightskyblue']),cbar=False,linewidths=0.1)
plt.show()

In [ ]:
tn=cm[0][0]
tp=cm[1][1]
fp=cm[0][1]
fn=cm[1][0]
total=tn+tp+fp+fn
correct_classify=((tn+tp)/total)*100
print('Correct Classify:',correct_classify)
incorrect_classify=((fp+fn)/total)*100
print('Incorrect Classify:',incorrect_classify)
specificity=tn/(tn+fp)
print('Specificity:',specificity)
sensitivity=tp/(tp+fn)
print('sensitivity:',sensitivity)